# Tutorial 1 — ASTER Mt. Rainier

A simple end-to-end ASP run. Single ASTER L1A scene to a 15 m DEM and a diagnostic PDF report.

We chose ASTER for the first tutorial because:

- The data is fully open. We use a Zenodo-hosted V003 archive; no Earthdata login.
- A single L1A scene contains both the nadir-looking and back-looking views, so there's no "finding a pair" step.
- The geometry is gentle (small B/H), so the DEM comes out smooth and easy to sanity-check.

## What we'll do

1. Download the L1A archive from Zenodo.
2. Run `aster2asp` to extract the nadir and back-looking imagery and camera files.
3. Run `parallel_stereo` (raw first pass) and `point2dem`.
4. Inspect the rough DEM with `asp_plot`.
5. Re-process with mapprojection for a higher-quality DEM.
6. Generate a one-page PDF report with `asp_plot`.

Scene: ASTER L1A acquired 2017-07-31 over Mt. Rainier, WA. UTM zone 10N (`EPSG:32610`).

Reference: [ASP's official ASTER example](https://stereopipeline.readthedocs.io/en/latest/examples/aster.html).


In [ ]:
DATA = "../data/aster_rainier"

!mkdir -p {DATA}


## 1. Download the scene

We use the V003 archive on Zenodo (preserved for tutorials after NASA decommissioned V003 in Dec 2025). The `download_aster.sh` script handles fetch + unzip; if the data is already there it's a no-op.

In [ ]:
!bash ../scripts/download_aster.sh {DATA}
!ls -1 {DATA}/dataDir/ | head

## 2. Sensor prep — `aster2asp`

`aster2asp` reads the L1A archive and writes out four files: a nadir-view image (`out-Band3N.tif`), a back-looking image (`out-Band3B.tif`), and an XML camera file for each. After this step the data is in the standard "image.tif + image.xml" pair that the rest of ASP works with.

In [ ]:
if Path(f"{DATA}/out-Band3N.tif").exists():
    print(f"{DATA}/out-Band3N.tif exists — skipping aster2asp. Delete out-Band3*.* to reprocess.")
else:
    !aster2asp {DATA}/dataDir -o {DATA}/out

!ls -1 {DATA}/out-Band3*.tif {DATA}/out-Band3*.xml


## 3. Stereo (no mapprojection) — first pass

Run `parallel_stereo` on the raw imagery to produce a reference DEM for the mapprojected pass below. This DEM gets downsampled to 200 m as the grid for the next pass, so quality matters less than speed.

- `-t aster` selects the ASTER session (knows about ASTER cameras).
- `--stereo-algorithm asp_bm`: plain block matching, fast.
- `--subpixel-mode 1`: parabolic subpixel refinement. Cheaper than mode 2 Bayes-EM, fine for a reference DEM.
- `--aster-use-csm` uses the Community Sensor Model camera.
- `--processes 8 --threads-multiprocess 1` matches the 8-core Codespace.


In [ ]:
if Path(f"{DATA}/out_stereo/run-DEM.tif").exists():
    print(f"{DATA}/out_stereo/run-DEM.tif exists — skipping first stereo pass. Delete {DATA}/out_stereo/ to reprocess.")
else:
    !parallel_stereo -t aster \
        --stereo-algorithm asp_bm \
        --subpixel-mode 1 \
        --aster-use-csm \
        --processes 8 --threads-multiprocess 1 \
        {DATA}/out-Band3N.tif {DATA}/out-Band3B.tif \
        {DATA}/out-Band3N.xml {DATA}/out-Band3B.xml \
        {DATA}/out_stereo/run

    !point2dem -r earth --auto-proj-center --errorimage {DATA}/out_stereo/run-PC.tif

!ls -lh {DATA}/out_stereo/run-DEM.tif


## 4. Inspect the rough DEM with `asp_plot`

`StereoPlotter.plot_detailed_hillshade()` shows the DEM as a hillshade with a 5 km-wide subset zoom on Mt. Rainier itself.


In [ ]:
from asp_plot.stereo import StereoPlotter

plotter = StereoPlotter(
    DATA,
    "out_stereo/",
    title="Rainier — first pass (no mapproj)",
)
plotter.plot_detailed_hillshade(subset_km=5)

ICESat-2 ATL06-SR vs the rough DEM.

In [ ]:
from utils import icesat2_check
icesat2_check(f"{DATA}/out_stereo/run-DEM.tif", directory=DATA)


## 5. Mapprojected pass

Use the rough DEM we just made, downsample it to 200 m, mapproject the input images onto it, and re-run stereo on the mapprojected pair. The disparity is now much smaller because the rough DEM is locally accurate.

In [ ]:
if Path(f"{DATA}/out_stereo_proj/run-DEM.tif").exists():
    print(f"{DATA}/out_stereo_proj/run-DEM.tif exists — skipping mapprojected pass. Delete {DATA}/out_stereo_proj/ to reprocess.")
else:
    # 200 m version of the rough DEM
    !point2dem -r earth --auto-proj-center --tr 200 \
        {DATA}/out_stereo/run-PC.tif -o {DATA}/out_stereo/run-200m

    # Mapproject both images onto the 200 m grid, output at 15 m
    !mapproject --tr 15 --aster-use-csm \
        {DATA}/out_stereo/run-200m-DEM.tif \
        {DATA}/out-Band3N.tif {DATA}/out-Band3N.xml \
        {DATA}/out-Band3N_proj.tif
    !mapproject --tr 15 --aster-use-csm \
        {DATA}/out_stereo/run-200m-DEM.tif \
        {DATA}/out-Band3B.tif {DATA}/out-Band3B.xml \
        {DATA}/out-Band3B_proj.tif

    # Stereo on the mapprojected images. The 200 m DEM is the reference grid.
    !parallel_stereo -t aster \
        --stereo-algorithm asp_mgm \
        --subpixel-mode 9 \
        --aster-use-csm \
        --processes 8 --threads-multiprocess 1 \
        {DATA}/out-Band3N_proj.tif {DATA}/out-Band3B_proj.tif \
        {DATA}/out-Band3N.xml {DATA}/out-Band3B.xml \
        {DATA}/out_stereo_proj/run \
        {DATA}/out_stereo/run-200m-DEM.tif

    !point2dem -r earth --auto-proj-center --errorimage {DATA}/out_stereo_proj/run-PC.tif

!ls -lh {DATA}/out_stereo_proj/run-DEM.tif


In [ ]:
plotter = StereoPlotter(
    DATA,
    "out_stereo_proj/",
    title="Rainier — mapprojected pass",
)
plotter.plot_detailed_hillshade(subset_km=5)

Same comparison against the mapprojected DEM.

In [ ]:
icesat2_check(f"{DATA}/out_stereo_proj/run-DEM.tif", directory=DATA)


## 6. Generate the full PDF report

Hand the whole directory to the `asp_plot` CLI to produce a one-page PDF: input scenes, disparity, hillshades, dh-vs-reference, and an ICESat-2 altimetry comparison.

We disable `--plot_geometry` because ASTER doesn't have the per-scene XML metadata that DigitalGlobe/Vantor provides for the stereo-geometry skyplot.


In [ ]:
if Path(f"{DATA}/out_stereo_proj/rainier_aster_report.pdf").exists():
    print(f"{DATA}/out_stereo_proj/rainier_aster_report.pdf exists — skipping asp_plot CLI. Delete it to regenerate.")
else:
    !asp_plot \
        --directory {DATA} \
        --stereo_directory out_stereo_proj \
        --reference_dem out_stereo/run-200m-DEM.tif \
        --plot_altimetry True \
        --plot_geometry False \
        --subset_km 5 \
        --report_filename rainier_aster_report.pdf


The report is at `data/aster_rainier/out_stereo_proj/rainier_aster_report.pdf`. Open it from the VS Code file tree.

## What's next

→ [Tutorial 2: WorldView-3 UCSD](02_worldview_ucsd.ipynb) adds bundle adjustment and ICESat-2 alignment.

→ For deeper ASTER processing (jitter correction, full bundle adjust), see [`asp_plot`'s ASTER notebooks](https://asp-plot.readthedocs.io/en/latest/examples/notebooks/aster_with_bundle_adjust_and_jitter_correction.html).
